In [ ]:
import json
import os
import socket
import datetime
from collections import Counter

# ─────────────────────────────────────────────────────────────
#  PATHS
# ─────────────────────────────────────────────────────────────

INCIDENT_LOG    = "incident_log.json"
BLOCKLIST_FILE  = "blocked_ips.txt"
QUARANTINE_FILE = "quarantine_hosts.txt"
REPORT_DIR      = "reports"

os.makedirs(REPORT_DIR, exist_ok=True)

# Baseline normal traffic values for comparison
BASELINE = {
    "flow_bytes_per_sec" : 50_000,
    "flow_packets_per_sec": 100,
    "syn_flag_count"     : 5,
}

# Local threat intel — maps attack types to known info
THREAT_INTEL = {
    "DDoS": {
        "full_name"  : "Distributed Denial of Service",
        "ttps"       : ["T1499 - Endpoint Denial of Service", "T1498 - Network Denial of Service"],
        "description": "Volumetric flood attack overwhelming target with traffic",
        "typical_ports": [80, 443, 53],
        "countermeasures": ["Rate limit inbound SYN", "Null route source IPs", "Enable scrubbing service"],
    },
    "PortScan": {
        "full_name"  : "Port Scanning",
        "ttps"       : ["T1046 - Network Service Discovery"],
        "description": "Attacker probing network for open ports and services",
        "typical_ports": ["any"],
        "countermeasures": ["Block source IP", "Enable port scan detection", "Review firewall rules"],
    },
    "Bot": {
        "full_name"  : "Botnet Activity",
        "ttps"       : ["T1071 - Application Layer Protocol", "T1090 - Proxy"],
        "description": "Compromised host communicating with C2 infrastructure",
        "typical_ports": [80, 443, 6667],
        "countermeasures": ["Quarantine host", "Block C2 domains", "Run malware scan"],
    },
    "FTP-Patator": {
        "full_name"  : "FTP Brute Force",
        "ttps"       : ["T1110 - Brute Force"],
        "description": "Automated credential stuffing attack against FTP service",
        "typical_ports": [21],
        "countermeasures": ["Lock FTP account", "Block source IP", "Enable MFA"],
    },
    "SSH-Patator": {
        "full_name"  : "SSH Brute Force",
        "ttps"       : ["T1110 - Brute Force"],
        "description": "Automated credential stuffing attack against SSH service",
        "typical_ports": [22],
        "countermeasures": ["Lock SSH account", "Block source IP", "Enable key-based auth only"],
    },
    "DoS Hulk": {
        "full_name"  : "DoS Hulk Attack",
        "ttps"       : ["T1499 - Endpoint Denial of Service"],
        "description": "HTTP flood attack generating unique URLs to bypass caching",
        "typical_ports": [80, 443],
        "countermeasures": ["Rate limit HTTP requests", "Enable CAPTCHA", "Block source IPs"],
    },
    "DoS GoldenEye": {
        "full_name"  : "DoS GoldenEye Attack",
        "ttps"       : ["T1499 - Endpoint Denial of Service"],
        "description": "HTTP DoS tool keeping connections alive to exhaust server",
        "typical_ports": [80, 443],
        "countermeasures": ["Limit concurrent connections", "Enable timeout rules"],
    },
    "DoS slowloris": {
        "full_name"  : "Slowloris Attack",
        "ttps"       : ["T1499 - Endpoint Denial of Service"],
        "description": "Slow HTTP attack holding connections open with partial requests",
        "typical_ports": [80, 443],
        "countermeasures": ["Limit connection timeout", "Enable request rate limiting"],
    },
    "DoS Slowhttptest": {
        "full_name"  : "Slow HTTP Test Attack",
        "ttps"       : ["T1499 - Endpoint Denial of Service"],
        "description": "Slow HTTP attack exhausting server connection pool",
        "typical_ports": [80, 443],
        "countermeasures": ["Tune server timeouts", "Enable WAF rules"],
    },
    "Infiltration": {
        "full_name"  : "Network Infiltration",
        "ttps"       : ["T1078 - Valid Accounts", "T1021 - Remote Services"],
        "description": "Attacker has gained internal network access",
        "typical_ports": [445, 3389, 22],
        "countermeasures": ["Isolate affected segment", "Force credential reset", "Enable EDR scan"],
    },
    "Heartbleed": {
        "full_name"  : "Heartbleed OpenSSL Exploit",
        "ttps"       : ["T1212 - Exploitation for Credential Access"],
        "description": "CVE-2014-0160 — memory leak via malformed TLS heartbeat",
        "typical_ports": [443],
        "countermeasures": ["Patch OpenSSL immediately", "Revoke and reissue certificates"],
    },
    "Web Attack Brute Force": {
        "full_name"  : "Web Application Brute Force",
        "ttps"       : ["T1110 - Brute Force"],
        "description": "Automated login attempts against web application",
        "typical_ports": [80, 443],
        "countermeasures": ["Lock account after failed attempts", "Enable CAPTCHA", "Block source IP"],
    },
    "Web Attack XSS": {
        "full_name"  : "Cross-Site Scripting",
        "ttps"       : ["T1059 - Command and Scripting Interpreter"],
        "description": "Malicious scripts injected into web application responses",
        "typical_ports": [80, 443],
        "countermeasures": ["Enable WAF XSS rules", "Sanitize input/output", "Enable CSP headers"],
    },
    "Web Attack Sql Injection": {
        "full_name"  : "SQL Injection",
        "ttps"       : ["T1190 - Exploit Public-Facing Application"],
        "description": "Malicious SQL queries injected to manipulate database",
        "typical_ports": [80, 443],
        "countermeasures": ["Enable WAF SQLi rules", "Use parameterized queries", "Restrict DB permissions"],
    },
    "BENIGN": {
        "full_name"  : "Normal Traffic",
        "ttps"       : [],
        "description": "No threat detected",
        "typical_ports": [],
        "countermeasures": [],
    },
}


# ─────────────────────────────────────────────────────────────
#  HELPER: load incident log safely
# ─────────────────────────────────────────────────────────────

def _load_log() -> list:
    if not os.path.exists(INCIDENT_LOG):
        return []
    with open(INCIDENT_LOG, "r") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []


def _save_log(log: list):
    with open(INCIDENT_LOG, "w") as f:
        json.dump(log, f, indent=2)


# ═════════════════════════════════════════════════════════════
#  FUNCTION LIBRARY
#  Each function returns a plain string result.
#  The orchestrator reads these results and decides next steps.
# ═════════════════════════════════════════════════════════════

def check_port_history(port: int) -> str:
    """Check how many times this port has been flagged in past incidents."""
    log   = _load_log()
    hits  = [i for i in log if str(i.get("port", "")) == str(port)]
    count = len(hits)
    if count == 0:
        return f"Port {port} has never been flagged before. Low historical risk."
    elif count <= 2:
        return f"Port {port} flagged {count} time(s) previously. Moderate historical risk."
    else:
        return f"Port {port} flagged {count} times previously. HIGH historical risk — recurring target."


def check_flow_rate(flow_bytes_per_sec: float) -> str:
    """Compare current flow rate to baseline normal traffic."""
    baseline = BASELINE["flow_bytes_per_sec"]
    ratio    = flow_bytes_per_sec / baseline if baseline > 0 else 0
    if ratio < 2:
        return f"Flow rate {flow_bytes_per_sec:,.0f} B/s is within normal range ({ratio:.1f}x baseline)."
    elif ratio < 10:
        return f"Flow rate {flow_bytes_per_sec:,.0f} B/s is elevated ({ratio:.1f}x baseline). Suspicious."
    else:
        return f"Flow rate {flow_bytes_per_sec:,.0f} B/s is {ratio:.0f}x above baseline. CRITICAL — volumetric attack confirmed."


def check_syn_count(syn_count: int) -> str:
    """Assess SYN flag count against normal baseline."""
    baseline = BASELINE["syn_flag_count"]
    ratio    = syn_count / baseline if baseline > 0 else 0
    if ratio < 3:
        return f"SYN count {syn_count} is within normal range."
    elif ratio < 20:
        return f"SYN count {syn_count} is elevated ({ratio:.0f}x baseline). Possible SYN flood."
    else:
        return f"SYN count {syn_count} is {ratio:.0f}x above baseline. SYN FLOOD CONFIRMED."


def lookup_threat_intel(attack_type: str) -> str:
    """Look up known threat intelligence for this attack type."""
    # fuzzy match — handle slight label variations
    matched = None
    for key in THREAT_INTEL:
        if key.lower() in attack_type.lower() or attack_type.lower() in key.lower():
            matched = key
            break

    if not matched:
        return f"No threat intel found for '{attack_type}'. Treat as unknown threat — escalate."

    intel = THREAT_INTEL[matched]
    ttps  = ", ".join(intel["ttps"]) if intel["ttps"] else "None recorded"
    steps = " | ".join(intel["countermeasures"]) if intel["countermeasures"] else "None"
    return (
        f"Attack: {intel['full_name']}\n"
        f"Description: {intel['description']}\n"
        f"MITRE TTPs: {ttps}\n"
        f"Countermeasures: {steps}"
    )


def get_similar_incidents(attack_type: str) -> str:
    """Find past incidents of the same attack type and what actions were taken."""
    log     = _load_log()
    similar = [i for i in log if attack_type.lower() in str(i.get("predicted_attack", "")).lower()]
    if not similar:
        return f"No previous incidents of type '{attack_type}' found in log."
    decisions = Counter(i.get("human_decision", "unknown") for i in similar)
    latest    = similar[-1]
    return (
        f"Found {len(similar)} previous '{attack_type}' incident(s).\n"
        f"Past decisions: {dict(decisions)}\n"
        f"Latest action taken: {latest.get('recommended_action', 'N/A')[:100]}"
    )


def calculate_severity(confidence: float, flow_bytes_per_sec: float, syn_count: int, port_hits: int) -> str:
    """Calculate overall severity score from 1-10."""
    score = 0
    score += min(confidence * 4, 4)                                    # max 4 pts from confidence
    score += min((flow_bytes_per_sec / BASELINE["flow_bytes_per_sec"]) * 0.5, 3)  # max 3 pts from flow
    score += min((syn_count / BASELINE["syn_flag_count"]) * 0.1, 2)   # max 2 pts from SYN
    score += min(port_hits * 0.5, 1)                                   # max 1 pt from history
    score  = round(min(score, 10), 1)

    if score >= 8:
        level = "CRITICAL"
    elif score >= 6:
        level = "HIGH"
    elif score >= 4:
        level = "MEDIUM"
    else:
        level = "LOW"

    return f"Severity score: {score}/10 — {level}"


def log_blocked_ip(ip: str, reason: str) -> str:
    """Add an IP to the blocklist file."""
    timestamp = datetime.datetime.now().isoformat()
    entry     = f"{timestamp} | {ip} | {reason}\n"
    with open(BLOCKLIST_FILE, "a") as f:
        f.write(entry)
    return f"IP {ip} added to blocklist ({BLOCKLIST_FILE}). Reason: {reason}"


def quarantine_host(host: str, reason: str) -> str:
    """Add a host to the quarantine list."""
    timestamp = datetime.datetime.now().isoformat()
    entry     = f"{timestamp} | {host} | {reason}\n"
    with open(QUARANTINE_FILE, "a") as f:
        f.write(entry)
    return f"Host {host} added to quarantine list ({QUARANTINE_FILE}). Reason: {reason}"


def flag_suspicious_ip(ip: str, attack_type: str, confidence: float) -> str:
    """Flag an IP as suspicious in the incident log."""
    log = _load_log()
    log.append({
        "timestamp"     : datetime.datetime.now().isoformat(),
        "type"          : "suspicious_ip_flag",
        "ip"            : ip,
        "attack_type"   : attack_type,
        "confidence"    : confidence,
    })
    _save_log(log)
    return f"IP {ip} flagged as suspicious for {attack_type} (confidence {confidence:.2%})."


def alert_admin(attack_type: str, severity: str, details: str) -> str:
    """Log an admin alert to the reports directory."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = os.path.join(REPORT_DIR, f"alert_{timestamp}.txt")
    content   = (
        f"ADMIN ALERT\n"
        f"{'='*40}\n"
        f"Time     : {datetime.datetime.now().isoformat()}\n"
        f"Attack   : {attack_type}\n"
        f"Severity : {severity}\n"
        f"Details  : {details}\n"
    )
    with open(filename, "w") as f:
        f.write(content)
    return f"Admin alert written to {filename}."


def log_clear(details: str) -> str:
    """Log a clear/benign traffic event."""
    log = _load_log()
    log.append({
        "timestamp": datetime.datetime.now().isoformat(),
        "type"     : "clear",
        "details"  : details,
    })
    _save_log(log)
    return f"Traffic marked as BENIGN and logged. {details}"


def write_incident_report(
    attack_type : str,
    confidence  : float,
    severity    : str,
    verdict     : str,
    trace       : list,
    features    : dict,
    human_decision: str = "pending"
) -> str:
    """Write a full structured incident report with the agent reasoning trace."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = os.path.join(REPORT_DIR, f"incident_{timestamp}.json")

    report = {
        "timestamp"      : datetime.datetime.now().isoformat(),
        "attack_type"    : attack_type,
        "confidence"     : confidence,
        "severity"       : severity,
        "verdict"        : verdict,
        "human_decision" : human_decision,
        "key_features"   : features,
        "reasoning_trace": trace,
    }

    with open(filename, "w") as f:
        json.dump(report, f, indent=2)

    # Also update incident log
    log = _load_log()
    log.append({
        "timestamp"       : report["timestamp"],
        "predicted_attack": attack_type,
        "confidence"      : confidence,
        "severity"        : severity,
        "recommended_action": verdict,
        "human_decision"  : human_decision,
        "port"            : features.get("Destination Port", "N/A"),
        "report_file"     : filename,
    })
    _save_log(log)

    return f"Incident report saved to {filename}."


# ─────────────────────────────────────────────────────────────
#  FUNCTION REGISTRY
#  Maps function names (strings) to actual callables.
#  The orchestrator uses this to dispatch by name.
# ─────────────────────────────────────────────────────────────

FUNCTION_REGISTRY = {
    "check_port_history"   : check_port_history,
    "check_flow_rate"      : check_flow_rate,
    "check_syn_count"      : check_syn_count,
    "lookup_threat_intel"  : lookup_threat_intel,
    "get_similar_incidents": get_similar_incidents,
    "calculate_severity"   : calculate_severity,
    "log_blocked_ip"       : log_blocked_ip,
    "quarantine_host"      : quarantine_host,
    "flag_suspicious_ip"   : flag_suspicious_ip,
    "alert_admin"          : alert_admin,
    "log_clear"            : log_clear,
    "write_incident_report": write_incident_report,
}


# ─────────────────────────────────────────────────────────────
#  ATTACK → DEFAULT FUNCTION MAP
#  Tells the orchestrator which functions are relevant
#  for each attack type as a starting point.
# ─────────────────────────────────────────────────────────────

ATTACK_FUNCTION_MAP = {
    "DDoS"                      : ["check_port_history", "check_flow_rate", "check_syn_count", "lookup_threat_intel", "calculate_severity", "log_blocked_ip", "alert_admin"],
    "PortScan"                  : ["check_port_history", "lookup_threat_intel", "calculate_severity", "flag_suspicious_ip", "alert_admin"],
    "Bot"                       : ["get_similar_incidents", "lookup_threat_intel", "calculate_severity", "quarantine_host", "alert_admin"],
    "FTP-Patator"               : ["check_port_history", "lookup_threat_intel", "calculate_severity", "flag_suspicious_ip", "alert_admin"],
    "SSH-Patator"               : ["check_port_history", "lookup_threat_intel", "calculate_severity", "flag_suspicious_ip", "alert_admin"],
    "DoS Hulk"                  : ["check_flow_rate", "lookup_threat_intel", "calculate_severity", "log_blocked_ip", "alert_admin"],
    "DoS GoldenEye"             : ["check_flow_rate", "lookup_threat_intel", "calculate_severity", "log_blocked_ip", "alert_admin"],
    "DoS slowloris"             : ["check_flow_rate", "lookup_threat_intel", "calculate_severity", "log_blocked_ip", "alert_admin"],
    "DoS Slowhttptest"          : ["check_flow_rate", "lookup_threat_intel", "calculate_severity", "log_blocked_ip", "alert_admin"],
    "Infiltration"              : ["get_similar_incidents", "lookup_threat_intel", "calculate_severity", "quarantine_host", "alert_admin"],
    "Heartbleed"                : ["lookup_threat_intel", "calculate_severity", "alert_admin"],
    "Web Attack Brute Force"    : ["check_port_history", "lookup_threat_intel", "calculate_severity", "flag_suspicious_ip", "alert_admin"],
    "Web Attack XSS"            : ["lookup_threat_intel", "calculate_severity", "alert_admin"],
    "Web Attack Sql Injection"  : ["lookup_threat_intel", "calculate_severity", "alert_admin"],
    "BENIGN"                    : ["log_clear"],
}


# ─────────────────────────────────────────────────────────────
#  QUICK TEST
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Testing function library...\n")

    print(check_port_history(80))
    print(check_flow_rate(1_500_000))
    print(check_syn_count(500))
    print(lookup_threat_intel("DDoS"))
    print(get_similar_incidents("DDoS"))
    print(calculate_severity(0.97, 1_500_000, 500, 3))
    print(log_blocked_ip("192.168.1.100", "DDoS attack source"))
    print(alert_admin("DDoS", "CRITICAL", "SYN flood on port 80"))
    print(write_incident_report(
        attack_type    = "DDoS",
        confidence     = 0.97,
        severity       = "CRITICAL",
        verdict        = "BLOCK",
        trace          = [{"thought": "test"}, {"action": "test"}],
        features       = {"Destination Port": 80, "Flow Bytes/s": 1500000},
        human_decision = "pending"
    ))
    print("\n[DONE] All functions working.")